# TP : Cyber + Matching Elasticsearch

Ce notebook regroupe **deux parties indépendantes** :

1. **Partie Cyber** — Tests de connectivité réseau (FQDN, DNS, TCP, HTTPS) sur des sites publics (badssl.com, space-match.net).
2. **Partie Matching** — Découverte du matching par vecteurs avec Elasticsearch (index, knn, script_score) et questions sur l'embedding.


## Préparation (commune)

- Ce dépôt cloné en local dans `TP_matching`.
- Un interpréteur Python pour la partie Matching ; pour la **partie Cyber**, les cellules sont en **PowerShell** (choisir le noyau PowerShell pour exécuter les cellules de la section 1, ou les copier dans un terminal PowerShell).

**Important :** pour la partie Matching, tu dois créer un fichier `.env` à la racine de `TP_matching` avec `ES_URL` et `API_KEY` qui pointent vers ton cluster Elasticsearch . Le notebook se connecte uniquement via ces variables.

**Si tu es dans un Codespace / environnement Linux sans PowerShell**, tu peux :
- soit lire la partie Cyber et l’exécuter sur un poste Windows avec PowerShell ;
- soit utiliser les **équivalents Linux** donnés juste en dessous (section "Notes pour Linux / Codespace").

In [ ]:
import sys
print("Python:", sys.version)

## 1. Partie Cyber : tests de connectivité 

### 1.1. Le FQDN (Fully Qualified Domain Name)

Pour joindre un serveur sur Internet, on utilise en général un **nom de domaine complet** (FQDN) (ex **badssl.com**) plutôt que son adresse IP. On va voir comment tester la résolution DNS, la connexion TCP et HTTPS.

### 1.2. Adresse IP et résolution DNS

Les machines sur le réseau sont identifiées par une **adresse IP**. Le **DNS** (Domain Name System) fait le lien entre un nom de domaine (FQDN) et une ou plusieurs adresses IP : c’est la **résolution DNS**. La commande **nslookup** permet d’obtenir l’IP associée à un nom.

Copiez/collez la commande ci-dessous dans un terminal **PowerShell** (elle n’est pas exécutée dans le notebook).

```powershell
  nslookup badssl.com
```
Ou sur linux : 

  ```bash
  sudo apt-get update
  sudo apt-get install dnsutils -y
  ```

> **Question :** à ton avis, que se passe‑t‑il si tu lances `nslookup` sur un nom inexistant (par exemple `does-not-exist.space-match.net`) ? Que peux‑tu en déduire sur le rôle du DNS dans la chaîne de connexion ?


<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <p>Pour un nom inexistant, <code>nslookup</code> renvoie une erreur de type
  <em>NXDOMAIN</em> ou un message indiquant qu’aucun enregistrement n’a été trouvé.</p>

  <p>On en déduit que le DNS est le <strong>premier maillon</strong> de la chaîne :
  si la résolution échoue, il est impossible de continuer (pas d’IP, donc pas de TCP/HTTPS).</p>

</details>

### 1.3. Connexion TCP et port 443

Une fois l’IP connue, il faut établir une **connexion TCP** vers le bon **port**. Pour le trafic HTTPS, le port standard est **443**. **Test-NetConnection** permet de vérifier si la machine peut ouvrir une connexion TCP vers ce port.

En **sécurité réseau**, une politique courante est de **bloquer** certains ports. Si on lance **Test-NetConnection** sans préciser le port, c’est le **ping (ICMP)** qui est testé ; un pare-feu peut autoriser le port 443 mais bloquer le ping.

```powershell
Test-NetConnection badssl.com -Port 443
```

Sans port = test ICMP (ping).

```powershell
Test-NetConnection badssl.com
```

Sur Linux: 

  ```bash
  timeout 1 bash -c '</dev/tcp/badssl.com/443 && echo PORT OPEN || echo PORT CLOSED'
  ```

> **Question :** si `Test-NetConnection badssl.com -Port 443` réussit mais que `Test-NetConnection badssl.com` (sans port) échoue, qu’est‑ce que cela t’indique sur les flux autorisés / bloqués par le pare‑feu entre ta machine et le serveur ?

<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <p>Si <code>Test-NetConnection badssl.com -Port 443</code> réussit mais
  <code>Test-NetConnection badssl.com</code> (sans port, donc ping ICMP) échoue :</p>

  <ul>
    <li>le port <strong>443/TCP est ouvert</strong> jusqu’au serveur (HTTPS possible),</li>
    <li>mais le trafic <strong>ICMP (ping) est bloqué</strong> par un pare‑feu (sur le chemin ou côté serveur).</li>
  </ul>

  <p>Donc le pare‑feu autorise le trafic applicatif HTTPS, mais bloque les pings.</p>

</details>

### 1.4. HTTPS et Invoke-WebRequest

Le protocole **HTTPS** assure le chiffrement et l’authentification du serveur (certificat). Pour tester qu’une URL HTTPS répond correctement en PowerShell : 

```powershell
Invoke-WebRequest -Uri "https://badssl.com" -UseBasicParsing
```

Sur Linux : 
  ```bash
  curl -v https://badssl.com
  ```

> **Question :** en observant les entêtes HTTP et les informations TLS retournées par `Invoke-WebRequest` ou `curl -v`, quels éléments te permettent de vérifier que tu parles bien au bon serveur et que la connexion est chiffrée (nom dans le certificat, protocole, code HTTP, etc.) ?

<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <p>Dans la sortie de <code>Invoke-WebRequest</code> ou <code>curl -v</code>, on peut vérifier :</p>

  <ul>
    <li>le <strong>protocole</strong> (<code>HTTPS</code>, TLSv1.2/1.3) → la connexion est chiffrée ;</li>
    <li>le <strong>certificat</strong> : CN / SAN qui doit contenir le FQDN (ex. <code>badssl.com</code>) ;</li>
    <li>le <strong>code HTTP</strong> (200, 301, 302…) indiquant que la requête a été acceptée.</li>
  </ul>

  <p>Si le nom dans le certificat ne correspond pas au FQDN, le navigateur afficherait une
  <strong>alerte de sécurité TLS</strong> (certificat non valide ou mauvais nom d’hôte).</p>

</details>

### 1.5. Exercices guidés : space-match.net

On reprend exactement la même démarche que pour `badssl.com`, mais cette fois sur le site `space-match.net`.

1. **Résolution DNS**  
   - Compare les IP de `space-match.net` et `staging.space-match.net` 
   - Question : **les deux noms pointent‑ils vers la même IP ou vers des IP différentes ? Qu’est‑ce que cela te suggère sur la façon dont les environnements (prod / staging) sont séparés ?**

2. **Connexion TCP / port 443**  
   - Teste la connexion TCP vers `space-match.net` sur le port **443**.  
   - Refais le test **sans préciser le port** (ping/ICMP).  
   - Question : **que déduis‑tu de l’état du pare‑feu ?**

3. **HTTPS et certificat**  
   - Lance une requête HTTPS (PowerShell : `Invoke-WebRequest`, Linux : `curl -v https://space-match.net`).  
   - Observe le CN / SAN du certificat et le code HTTP de réponse.  
   - Questions :  
     - **le certificat présenté correspond‑il bien au nom `space-match.net` ?**  
     - **que verrais‑tu dans le navigateur si le certificat ne correspondait pas au FQDN (erreur TLS typique) ?**


<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <p><strong>DNS</strong> : <code>space-match.net</code> et
  <code>staging.space-match.net</code> ont des IPs différentes ce qui suggère une
  <strong>séparation claire des environnements</strong> (prod vs staging).</p>

  <p><strong>TCP 443</strong> : les deux réussisent car le ping n'est pas filtré.</p>

  <p><strong>HTTPS</strong> : le certificat doit contenir <code>space-match.net</code>
  dans le CN ou les SAN. Sinon, une erreur TLS (certificat pour un autre nom) serait affichée.</p>

</details>

## 2. Partie Matching : cluster Elasticsearch et index de produits

Pour la suite, on travaille avec un **cluster Elasticsearch défini dans `.env`** (par exemple un déploiement Elastic Cloud). Le notebook utilise l’URL et la clé API fournies dans ce fichier.


### 2.1 Mapping de l’index `temp_tp_matching`

Avant de lancer des requêtes, il faut savoir **à quoi ressemblent les documents** dans l’index. En Elasticsearch, cette description s’appelle le **mapping** : c’est le « schéma logique » des champs (texte, mot‑clé, vecteur, date, etc.).

Pour l’index `temp_tp_matching`, les champs importants sont :
- `product` : description textuelle du produit (`text` + sous‑champ `product.keyword`) ;
- `category` : catégorie du produit (`keyword`), pratique pour filtrer (par ex. ne garder que les `shoes`) ;
- `semantic_vector` : vecteur dense de dimension 4 (`dense_vector`) **indexé** pour la recherche KNN (similarité cosinus) ;
- `custom_vector` : valeur numérique (`float`, non indexée) qui illustre un signal stocké mais non requêtable directement par filtre (utile dans un `script_score`) ;
- `lastmodificationdate` : date de dernière modification du document (`date`).

Ce mapping est défini dans `install/es_data/mapping_products.json` et appliqué lors de la création de l’index par `install/bulk_temp_index`.

```json
{
  "mappings": {
    "properties": {
      "product": {
        "type": "text",
        "fields": {
          "keyword": {
            "type": "keyword",
            "ignore_above": 256
          }
        }
      },
      "category": {
        "type": "keyword"
      },
      "semantic_vector": {
        "type": "dense_vector",
        "dims": 4,
        "index": true,
        "similarity": "cosine"
      },
      "custom_vector": {
        "type": "float",
        "index": false
      },
      "lastmodificationdate": {
        "type": "date"
      }
    }
  }
}
```
**Question :**

On veut créer un index Elasticsearch avec :

- un champ textuel `description` (texte libre, avec un sous‑champ `description.keyword` pour les agrégations),
- un champ vectoriel `vector` (embeddings) de dimension 10, de type `dense_vector`, indexé pour pouvoir faire des requêtes KNN en similarité cosinus.

>**Écris le mapping JSON complet pour cet index.**


<details>
  <summary><strong>Afficher une solution possible</strong></summary>

  ```python
  "mappings": {
    "properties": {
      "description": {
        "type": "text",
        "fields": {
          "keyword": {
            "type": "keyword",
            "ignore_above": 256
          }
        }
      },
      "vector": {
        "type": "dense_vector",
        "dims": 10,
        "index": true,
        "similarity": "cosine"
      }
    }
  }
```

</details>

### 2.1 Client Elasticsearch synchrone puis asynchrone

Pour parler à Elasticsearch depuis Python, on utilise le **client officiel** du package `elasticsearch`. En mode **synchrone**, on crée un objet `Elasticsearch(...)` et chaque appel (par exemple `es.count(...)` ou `es.search(...)`) **bloque** jusqu’à ce que le cluster ait répondu : le code s’exécute donc étape par étape, comme une conversation téléphone où l’on attend la réponse avant de continuer. En mode **asynchrone**, on utilise au contraire `AsyncElasticsearch(...)` avec les mots‑clés `async` / `await` et une boucle `asyncio` : les requêtes deviennent des tâches non bloquantes que l’on peut lancer, laisser tourner en arrière‑plan, puis `await` plus tard, ce qui permet de gérer efficacement plusieurs appels réseau en parallèle sans figer le reste du programme.


In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

# Charger ES_URL et ES_API_KEY directement depuis .env
env_path = Path(".env")
load_dotenv(dotenv_path=env_path)
ES_URL = os.getenv("ES_URL")
ES_API_KEY = os.getenv("API_KEY")
if not ES_URL or not ES_API_KEY:
    raise RuntimeError("ES_URL ou API_KEY manquant dans .env")

# Client synchrone Elasticsearch
es_sync = Elasticsearch(
    hosts=[ES_URL.rstrip("/")],
    api_key=ES_API_KEY,
    request_timeout=60,
    max_retries=3,
    retry_on_timeout=True,
)

# Exemple : compter les documents de l’index temp_tp_matching
count_resp = es_sync.count(index="temp_tp_matching")
print("Nombre de documents (synchrone) :", count_resp.get("count"))
es_sync.close()

En asynchrone, on utilise `AsyncElasticsearch` et `asyncio` :

- le client s’instancie de la même façon (mêmes paramètres : `hosts`, `api_key`, timeouts, retries) ;
- les méthodes deviennent **asynchrones** → on les appelle avec `await` ;
- il faut une fonction `async def ...` et une boucle d’événement (par ex. `asyncio.run(...)`).

Complète le code ci‑dessous pour écrire l’équivalent **asynchrone** du comptage de documents.

In [ ]:
rom pathlib import Path
import os
from dotenv import load_dotenv
from elasticsearch import AsyncElasticsearch
# TODO Charger ES_URL et ES_API_KEY directement depuis .env

async def async_count_documents() -> None:
    # Client asynchrone Elasticsearch
    es_async = 


    try:
        # Exemple : compter les documents de l’index temp_tp_matching
        resp = 
        print("Nombre de documents (asynchrone) :", resp.get("count"))
    finally:
        await es_async.close()
# En notebook, on peut appeler directement la coroutine avec await
await 

<details>
  <summary><strong>Afficher une solution possible (client asynchrone)</strong></summary>
  
  ```python
    from pathlib import Path
    import os
    from dotenv import load_dotenv
    from elasticsearch import AsyncElasticsearch

    # Charger ES_URL et ES_API_KEY directement depuis .env
    env_path = Path(".env")
    load_dotenv(dotenv_path=env_path)
    ES_URL = os.getenv("ES_URL")
    ES_API_KEY = os.getenv("API_KEY")
    if not ES_URL or not ES_API_KEY:
        raise RuntimeError("ES_URL ou API_KEY manquant dans .env")

    async def async_count_documents() -> None:
        # Client asynchrone Elasticsearch
        es_async = AsyncElasticsearch(
            hosts=[ES_URL.rstrip("/")],
            api_key=ES_API_KEY,
            request_timeout=60,
            max_retries=3,
            retry_on_timeout=True,
        )
        try:
            # Exemple : compter les documents de l’index temp_tp_matching
            resp = await es_async.count(index="temp_tp_matching")
            print("Nombre de documents (asynchrone) :", resp.get("count"))
        finally:
            await es_async.close()

    # En notebook (Jupyter/Cursor), on peut appeler directement la coroutine avec await
    await async_count_documents()
  ```
</details>

### 2.3. Requête KNN simple

**Idée** : chaque produit est représenté par un **vecteur** (`semantic_vector` de dimension 4 dans l’index `temp_tp_matching`). On donne aussi un vecteur de requête, et Elasticsearch renvoie les *k* produits dont le vecteur est le plus proche (similarité cosinus).

Concrètement :
- l’index s’appelle **`temp_tp_matching`** et contient un champ `semantic_vector` (défini dans `install/es_data/mapping_products.json`) ;
- on envoie une requête `POST` sur `"{ES_URL}/temp_tp_matching/_search"` avec un bloc `knn` dans le corps JSON qui précise :
  - `"field": "semantic_vector"` (le champ vecteur à utiliser) ;
  - `"query_vector"` (le vecteur que l’on fournit) ;
  - `"k"` (le nombre de voisins à renvoyer) ;
  - `"num_candidates"` (le nombre de candidats parcourus avant de garder les `k` meilleurs, pour gérer le compromis performance / qualité).

La réponse JSON contient `hits.hits` : pour chaque document, on peut afficher par exemple le texte du produit (`_source["product"]`) et son `_score` (plus le score est élevé, plus le document est proche du vecteur de requête).

Sous le capot, Elasticsearch utilise des index de vecteurs (graphe **HNSW**, etc.) pour approximer ces plus proches voisins (**ANN, Approximate Nearest Neighbors**). Pour aller plus loin :
- Docs `dense_vector` :  <https://www.elastic.co/guide/en/elasticsearch/reference/current/dense-vector.html>  
- Docs recherche KNN (`knn` dans `_search`) : <https://www.elastic.co/guide/en/elasticsearch/reference/current/knn-search.html>  
- Vue d’ensemble de la recherche vectorielle / sémantique : <https://www.elastic.co/guide/en/elasticsearch/reference/current/semantic-search.html>


In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

# Charger ES_URL et ES_API_KEY
env_path = Path(".env")
load_dotenv(dotenv_path=env_path)
ES_URL = os.getenv("ES_URL")
ES_API_KEY = os.getenv("API_KEY")
if not ES_URL or not ES_API_KEY:
    raise RuntimeError("ES_URL ou API_KEY manquant dans .env")

# 1) Recréer un client pour cette requête
es = Elasticsearch(
    hosts=[ES_URL.rstrip("/")],
    api_key=ES_API_KEY,
    request_timeout=60,
    max_retries=3,
    retry_on_timeout=True,
)

try:

    ### Creation de la query 
    query_vector = [1.0, 0.0, 0.0, 0.0]
    knn_body = {
        "knn": {
            "field": "semantic_vector",
            "query_vector": query_vector,
            "k": 4,
            "num_candidates": 10,
        }
    }

    resp = es.search(index="temp_tp_matching", body=knn_body)
    for hit in resp["hits"]["hits"]:
        print(hit["_source"]["product"], "— score:", hit["_score"])
finally:
    es.close()



> **Question 1 – Comprendre le résultat**  
  Quand tu regardes la réponse JSON de la requête KNN, que représentent les champs `_score` et `_source["product"]` pour chaque document retourné ?

<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <p>Dans <code>hits.hits</code> :</p>

  <ul>
    <li><code>_source["product"]</code> contient la description textuelle du produit telle qu’elle a été indexée.</li>
    <li><code>_score</code> correspond au score de similarité calculé par Elasticsearch entre le <code>semantic_vector</code> du document et le <code>query_vector</code>. Plus le score est élevé, plus le document est aligné enterme d'angle vis à vis de la query (cas de la métrique cosine).</li>
  </ul>

</details>

> **Question 2 – Effet de la requête sur les résultats**  
  Que se passe‑t‑il si tu modifies légèrement le vecteur de requête, par exemple en passant de <code>[1.0, 0.0, 0.0, 0.0]</code> à <code>[0.9, 0.1, 0.0, 0.0]</code> ? Qu’attends‑tu comme impact sur les documents retournés et leurs scores ?

<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <p>En changeant un peu le vecteur de requête, on déplace légèrement le « point » dans l’espace vectoriel :</p>

  <ul>
    <li>les <strong>produits les plus proches</strong> resteront souvent similaires (même type de produits), mais l’ordre ou les scores peuvent changer ;</li>
  </ul>


</details>

> **Question 3 – Rôle de `k` et `num_candidates`**  
  Dans le bloc `knn`, à quoi servent les paramètres `k` et `num_candidates` ? Que peut‑il se passer si l’on fixe un `num_candidates` très petit par rapport à `k` ?

<details>
  <summary><strong>Voir une solution possible</strong></summary>

  <ul>
    <li><code>k</code> : nombre final de voisins que l’on veut récupérer (la taille de la liste de résultats principaux).</li>
    <li><code>num_candidates</code> : nombre maximum de documents que l’algorithme va explorer avant de ne garder que les <code>k</code> meilleurs.</li>
  </ul>

  <p>Si <code>num_candidates</code> est très petit par rapport à <code>k</code> :</p>

  <ul>
    <li>la requête peut être plus rapide (moins de candidats à parcourir) ;</li>
    <li>mais on risque de <strong>ne pas voir certains bons voisins</strong> parce qu’ils n’ont jamais été considérés comme candidats ;</li>
    <li>les <code>k</code> résultats finaux peuvent donc être moins « optimaux » du point de vue de la similarité.</li>
  </ul>

  <p>En pratique, on choisit <code>num_candidates</code> pour trouver un compromis entre <strong>qualité des résultats</strong> et <strong>temps de réponse</strong>.</p>

</details>

### 2.4 Requête KNN avec filtre

On reprend la requête KNN sur le champ vectoriel `semantic_vector`, mais cette fois en ajoutant un **filtre classique** sur la catégorie du produit.  
L’idée est la suivante :

- **Partie KNN**  
  - on fournit un vecteur de requête (par exemple `[1.0, 0.0, 0.0, 0.0]`) dans `query_vector` ;
  - Elasticsearch cherche les documents les plus proches de ce vecteur dans le champ `semantic_vector` (type `dense_vector`, `index: true`, `similarity: "cosine"`).

- **Partie filtre**  
  - on ajoute un filtre `term` sur le champ `category` (type `keyword`) pour ne garder qu’un sous‑ensemble de documents (par exemple uniquement la catégorie `"shoes"`) ;
  - le filtre réduit l’ensemble des candidats avant de calculer la similarité vectorielle.

Concrètement, la requête KNN avec filtre combine donc :
1. un **espace vectoriel** pour mesurer la similarité (`semantic_vector`) ;
2. un **filtre booléen** sur un champ discret (`category`) pour restreindre la recherche à un segment de l’index (ex. toutes les chaussures).


In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

env_path = Path(".env")
load_dotenv(dotenv_path=env_path)
ES_URL = os.getenv("ES_URL")
ES_API_KEY = os.getenv("API_KEY")
if not ES_URL or not ES_API_KEY:
    raise RuntimeError("ES_URL ou API_KEY manquant dans .env")

# Recréer un client pour cette requête
es = Elasticsearch(
    hosts=[ES_URL.rstrip("/")],
    api_key=ES_API_KEY,
    request_timeout=60,
    max_retries=3,
    retry_on_timeout=True,
)

try:
    query_vector = [1.0, 0.0, 0.0, 0.0]
    ### Creation de la query 
    knn_with_filter_body = {
        "knn": {
            "field": "semantic_vector",
            "query_vector": query_vector,
            "k": 4,
            "num_candidates": 10,
            "filter": {
                "term": {"category": "shoes"}  # ex : ne garder que les chaussures
            },
        }
    }
    resp = es.search(index="temp_tp_matching", body=knn_with_filter_body)
    for hit in resp["hits"]["hits"]:
        print(hit["_source"]["product"], "— score:", hit["_score"])
finally:
    es.close()

**Exercice – filtrer sur plusieurs catégories**

On repart de la requête KNN avec filtre vue précédemment (filtre `category: "shoes"`).  
Modifie le corps de la requête pour ne garder que les produits dont la catégorie est **`bag` ou `clothes`**.

Autrement dit, ta requête doit faire :

- une recherche KNN sur `semantic_vector` ;
- **ET** ne garder que les documents avec `category` dans `{bag, clothes}`.

<details>
  <summary><strong>Indice 1 – type de filtre</strong></summary>

  Pour filtrer sur plusieurs valeurs d’un champ de type <code>keyword</code>, tu peux utiliser soit :

  - une requête booléenne avec plusieurs <code>term</code> dans un <code>should</code> (et un <code>minimum_should_match</code>), ou
  - une requête <code>terms</code> avec un tableau de valeurs.

  La doc des bool queries :  
  https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl-bool-query.html
  https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl-terms-query.html
</details>

<details>
  <summary><strong>Indice 2 – syntaxe possible</strong></summary>

  Une façon simple de filtrer sur <code>bag</code> ou <code>clothes</code> est d’utiliser une clause <code>filter</code>
  de type <code>terms</code> à l’intérieur du bloc <code>knn</code> :

  - champ : <code>"category"</code>  
  - valeurs : <code>["bag", "clothes"]</code>.

  À toi de l’insérer au bon endroit dans le corps JSON de la requête.
</details>


<details>
  <summary><strong>Solution avec l’opérateur <code>terms</code> </strong></summary>

```python
knn_with_filter_body = {
    "knn": {
        "field": "semantic_vector",
        "query_vector": query_vector,
        "k": 4,
        "num_candidates": 10,
        "filter": {
            "terms": {
                "category": ["bag", "clothes"]
            }
        },
    }
}
```
</details>

<details>
  <summary><strong>Solution avec l’opérateur <code>should</code></strong></summary>


```python
knn_with_filter_body = {
    "knn": {
        "field": "semantic_vector",
        "query_vector": query_vector,
        "k": 4,
        "num_candidates": 10,
        "filter": {
            "bool": {
                "should": [
                    {"term": {"category": "bag"}},
                    {"term": {"category": "clothes"}},
                ],
                "minimum_should_match": 1
            }
        },
    }
}
```
</details>



## 3. Requête de matching avec `script_score`

Jusqu’ici, la similarité était calculée par Elasticsearch avec une **métrique standard** (par exemple la similarité cosinus) directement sur un champ `dense_vector` comme `semantic_vector`. Cela couvre beaucoup de cas, mais parfois on veut aller **plus loin** :

- combiner plusieurs signaux (vecteur sémantique + score métier, clics, popularité, etc.) ;
- appliquer une formule de scoring personnalisée qui ne correspond pas exactement à une distance standard ;
- ajuster dynamiquement le poids de certaines dimensions selon le contexte métier.

Pour cela, on peut :

1. stocker notre signal numérique additionnel dans un champ comme `custom_vector` (dans notre mapping, ce champ illustre justement un score / vecteur métier non indexé en KNN) ;
2. utiliser une requête `script_score` pour écrire **nous‑mêmes** la formule de score avec un script Painless.

On utilise donc un script **Painless** (le langage de script embarqué d’Elasticsearch) pour calculer un score à partir :

- d’un vecteur d’entrée `params.input_vector` fourni dans la requête,
- du vecteur ou des valeurs stockées dans le document, par exemple `doc['custom_vector']`,
- (optionnel) d’un vecteur de poids `params.weights` si l’on veut pondérer certaines dimensions.

L’idée générale d’un `script_score` est :

1. calculer une **distance** ou une **similarité** entre l’entrée et ce qui est stocké (par exemple une distance euclidienne),
2. transformer cette distance en **score** exploitable par Elasticsearch, par exemple :  
   \( score = \frac{1}{1 + distance} \).

Le script est écrit en Painless, une syntaxe proche de Java, qui permet d’accéder aux champs de `doc` et aux paramètres `params`. Pour plus de détails sur Painless et `script_score` :

- Docs Painless : <https://www.elastic.co/guide/en/elasticsearch/painless/current/painless-guide.html>  
- Docs `script_score` : <https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl-script-score-query.html>

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from elasticsearch import Elasticsearch

# Charger la config depuis .env
env_path = Path(".env")
load_dotenv(dotenv_path=env_path)
ES_URL = os.getenv("ES_URL")
ES_API_KEY = os.getenv("API_KEY")
if not ES_URL or not ES_API_KEY:
    raise RuntimeError("ES_URL ou API_KEY manquant dans .env")

# Script Painless pour calculer un score à partir de custom_vector (L1 sans pondération)

script = {
    "source": """
double l1_distance = 0.0;
for (int i = 0; i < params.input_array.length; i++) {
  double diff = params.input_array[i] - doc['custom_vector'][i];
  l1_distance += Math.abs(diff);
}
double score = 1.0 / (1.0 + l1_distance);
return score;
""",
    "params": {
        "input_array": [1.0, 0.5, 0.5, 0.3],
    },
}

query_body = {
    "size": 4,
    "query": {
        "script_score": {
            "query": {"match_all": {}},
            "script": script,
        }
    },
}

# Client ES
es = Elasticsearch(
    hosts=[ES_URL.rstrip("/")],
    api_key=ES_API_KEY,
    request_timeout=60,
    max_retries=3,
    retry_on_timeout=True,
)

try:
    resp = es.search(index="temp_tp_matching", body=query_body)
    for hit in resp["hits"]["hits"]:
        src = hit.get("_source", {})
        print(src.get("product"), "| custom_vector:", src.get("custom_vector"), "— score:", hit.get("_score"))
finally:
    es.close()

**Question – Prioriser certaines dimensions du vecteur**
Modifie le script Painless précédent pour que les **deux premières dimensions** du vecteur `custom_vector` comptent **3 fois plus** que les deux suivantes dans le calcul de la distance (et donc du score).  

<details>
  <summary><strong>Indice – comment intégrer les poids ?</strong></summary>
  <p>Tu peux ajouter un tableau <code>weights</code> dans les <code>params</code> :</p>
  <pre><code>"params": {
    "input_array": input_array,
    "weights": [3.0, 3.0, 1.0, 1.0]
}</code></pre>
  <p>Puis, dans la boucle Painless :</p>
  <ul>
    <li>récupérer le poids correspondant avec <code>params.weights[i]</code> ;</li>
    <li>additionner <code>params.weights[i] * Math.abs(diff)</code> dans <code>l1_distance</code> au lieu de juste <code>Math.abs(diff)</code>.</li>
  </ul>
</details>
<details>
  <summary><strong>Solution possible</strong></summary>
  <p>Voici une version pondérée du script :</p>
  
  ```python
  script = {
      "source": """
  double l1_distance = 0.0;
  for (int i = 0; i < params.input_array.length; i++) {
    double diff = params.input_array[i] - doc['custom_vector'][i];
    double weight = params.weights[i];
    l1_distance += weight * Math.abs(diff);
  }
  double score = 1.0 / (1.0 + l1_distance);
  return score;
  """,
      "params": {
          "input_array": [1.0, 0.5, 0.5, 0.3],
          "weights": [3.0, 3.0, 1.0, 1.0],
      },
  }

## 4. Comment construire les vecteurs de matching ?

Jusqu’ici, on a supposé que les vecteurs (`semantic_vector`, `custom_vector`) existaient déjà dans l’index.  
On a appris à :

- faire de la recherche **KNN** sur un champ `dense_vector` (`semantic_vector`) ;
- écrire un **`script_score`** en Painless pour redéfinir nous‑mêmes la fonction de score à partir de vecteurs.

La question suivante naturelle est : **comment construire ces vecteurs ?**

L’idée générale est de transformer les informations d’un produit (catégorie, prix, popularité, etc.) en un **vecteur numérique** qui encode ce que l’on veut mesurer : similarité de goût, compatibilité, priorité métier, etc.

---

### Q1 – Comment construire un vecteur à partir de données métier (catégories, prix, etc.) ?

<details>
  <summary><strong>Voir une réponse</strong></summary>

  <p>Avant d'utiliser des modèles complexes (transformers…), on peut construire des <em>embeddings</em> « à la main » à partir de <strong>features catégorielles</strong> (ex. <code>category</code>, <code>brand</code>) et <strong>réelles</strong> (ex. <code>price</code>, <code>rating</code>, <code>nb_clicks</code>).</p>

  <p>Exemple simplifié : pour chaque produit on a <code>category</code>, <code>price</code>, <code>rating</code>. On forme un vecteur de dimension 4 : dimensions 0–1 = encodage catégoriel (ex. is_shoes, is_bag), dimension 2 = prix normalisé, dimension 3 = note normalisée. Ex. <code>custom_vector = [is_shoes, is_bag, price_normalized, rating_normalized]</code>. Le matching compare alors catégorie et valeurs continues en même temps. C'est l'idée derrière <code>custom_vector</code> dans notre index.</p>

</details>

---

### Q2 – Pourquoi standardiser ou normaliser les features avant de les mettre dans un vecteur ?

<details>
  <summary><strong>Voir une réponse</strong></summary>

  <p>Problème d'<strong>échelle</strong> : le prix peut aller de 10 à 1000, la note de 1 à 5, un binaire de 0 à 1. Si on utilise des vecteurs bruts, la dimension « prix » domine la distance (L1, euclidienne…).</p>
  <p>On ramène donc chaque dimension sur une échelle comparable :</p>
  <ul>
    <li><strong>Normalisation</strong> (min‑max) : <code>x_norm = (x - x_min) / (x_max - x_min)</code> dans [0, 1].</li>
    <li><strong>Standardisation</strong> (centrer‑réduire) : <code>x_std = (x - mu) / sigma</code> (mu = moyenne, sigma = écart‑type).</li>
  </ul>
  <p>En standardisant, la moyenne de chaque dimension est ~0 et la variance ~1, ce qui donne un poids comparable à chaque dimension dans la distance.</p>

</details>

---

### Q3 – Comment standardiser une feature réelle concrètement ?

<details>
  <summary><strong>Voir une réponse</strong></summary>

  <p>Pour une dimension (ex. le prix) :</p>
  <ol>
    <li><strong>Moyenne</strong> : <code>mu = (1/N) * sum(x_i)</code> pour i de 1 à N.</li>
    <li><strong>Variance puis écart‑type</strong> : <code>variance = (1/(N-1)) * sum((x_i - mu)^2)</code>, puis <code>sigma = sqrt(variance)</code>.</li>
    <li><strong>Standardisation</strong> : <code>x_std = (x - mu) / sigma</code>.</li>
  </ol>
  <p>Avec des millions de produits, on ne veut pas recalculer mu et sigma à chaque nouvel ajout en repassant sur toute la base. Il faut un algorithme qui met à jour ces statistiques <strong>au fil de l'eau</strong> (voir Q4).</p>

</details>

---

### Q4 – Comment mettre à jour la moyenne et la variance à chaque nouvel ajout sans tout recalculer ?

<details>
  <summary><strong>Voir une réponse</strong></summary>

  <p>On veut : à chaque nouveau document avec une valeur <code>x_new</code> sur une dimension, mettre à jour <code>mu</code> et <code>sigma</code> sans relire tout l'historique.</p>
  <p>L'algorithme <strong>online</strong> de <strong>Welford</strong> fait exactement ça. Pour une dimension, il maintient :</p>
  <ul>
    <li><code>count</code> : nombre d'échantillons vus ;</li>
    <li><code>mean</code> : moyenne courante ;</li>
    <li><code>M2</code> : somme des carrés des écarts (intermédiaire pour la variance).</li>
  </ul>
  <p>À chaque nouvelle valeur <code>x</code> :</p>
  <pre><code>count = count + 1
delta = x - mean
mean = mean + delta / count
delta2 = x - mean
M2 = M2 + delta * delta2

if count > 1:
    variance = M2 / (count - 1)
    std = sqrt(variance)</code></pre>
  <p>On peut ainsi mettre à jour les stats à l'indexation de chaque document, puis utiliser mean et std pour standardiser les nouvelles valeurs (ex. dans <code>pk_spacematch_backend</code>).</p>

</details>

## BONUS — Question avancée

Expliquez pourquoi le **phénomène de concentration des volumes sur les bords** en haute dimension explique en partie la **perte de performance des méthodes de matching** quand on utilise des vecteurs de très grande dimension.

<details>
  <summary><strong>Voir une solution possible</strong></summary>

  **1. Le phénomène de concentration des volumes sur les bords**

  En dimension *d*, une boule de rayon 1 a un volume proportionnel à 1, mais la plus grande partie de ce volume se situe dans une **coquille** proche de la surface. Par exemple, pour une boule de rayon 1, le volume contenu dans la boule intérieure de rayon (1 − ε) (avec ε petit, ex. 0,1) est (1 − ε)^d : en grande dimension ce rapport tend très vite vers 0. Donc « presque tout » le volume est dans la fine couche entre les rayons (1 − ε) et 1. On dit que le volume se **concentre sur les bords** : il n’y a presque plus de masse au centre.

  **2. Conséquence sur les distances**

  Quand on tire des points « au hasard » dans une telle région (ou dans un cube en haute dimension), les **distances entre paires de points** (euclidienne, ou après normalisation, cosinus) deviennent très proches les unes des autres : la variance des distances diminue par rapport à la distance moyenne. En pratique, le « plus proche voisin » n’est plus beaucoup plus proche que le 10ᵉ ou le 100ᵉ : les scores de similarité se **tassent**, ce qui rend le classement par similarité peu discriminant.

  **3. Impact sur les méthodes de matching (KNN / ANN)**

  - Les algorithmes de **recherche des k plus proches voisins** (KNN) ou **approximate nearest neighbors** (ANN, ex. HNSW utilisé par Elasticsearch) reposent sur l’idée que les « vrais » plus proches voisins ont une distance nettement plus faible que le reste. En haute dimension, cette hypothèse est moins vraie : beaucoup de points ont des distances comparables.
  - Les **structures d’index** (graphes, partitions) deviennent moins efficaces pour filtrer les candidats, car la géométrie ne sépare plus bien les proches des lointains. Il faut explorer plus de noeuds (augmentation de `num_candidates`, etc.), ce qui dégrade le **coût de calcul** sans toujours améliorer la **qualité** du ranking.
  - En résumé : **qualité** du matching (pertinence du classement) et **performance** (temps, mémoire) se dégradent quand la dimension des vecteurs devient trop grande.

  **4. Pistes pour limiter le problème**

  - **Réduction de dimension** : PCA, SVD, ou modèles d’embeddings avec une dimension plus faible (ex. 256–768 au lieu de 3000+) pour garder une géométrie plus « utile ».
  - **Métriques et normalisation** : la similarité cosinus sur vecteurs normalisés est souvent plus robuste que la distance euclidienne brute en haute dimension ; le choix de la similarité dans le mapping (`similarity: "cosine"`) va dans ce sens.
  - **Dimension des champs `dense_vector`** : en pratique, éviter des dimensions excessives dans les index vectoriels (Elasticsearch recommande des ordres de grandeur raisonnables selon le type d’index).

</details>


**Sources :**  
- Bellman (1961), *Adaptive Control Processes* — « curse of dimensionality ».  
- Aggarwal et al., « On the Surprising Behavior of Distance Metrics in High Dimensional Space », ICDT 2001.  
- Blum, Hopcroft, Kannan, *Foundations of Data Science* (2020), ch. haute dimension — [cs.cornell.edu/jeh/book.pdf](https://www.cs.cornell.edu/jeh/book.pdf).  
- Weber et al., « A Quantitative Analysis… for Similarity-Search Methods in High-Dimensional Spaces », VLDB 1998.  
- Elasticsearch : [dense_vector](https://www.elastic.co/guide/en/elasticsearch/reference/current/dense-vector.html), [kNN search](https://www.elastic.co/guide/en/elasticsearch/reference/current/knn-search.html).